In [1]:
import pandas as pd
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [2]:
df = pd.read_csv('../datasets/diabetes/diabetes_prediction_dataset.csv')
df.head()

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


In [3]:
print(df.shape)
print(df.info())
print(df.isnull().sum())
print(df.duplicated().sum())
print(df['diabetes'].value_counts())

(100000, 9)
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   gender               100000 non-null  str    
 1   age                  100000 non-null  float64
 2   hypertension         100000 non-null  int64  
 3   heart_disease        100000 non-null  int64  
 4   smoking_history      100000 non-null  str    
 5   bmi                  100000 non-null  float64
 6   HbA1c_level          100000 non-null  float64
 7   blood_glucose_level  100000 non-null  int64  
 8   diabetes             100000 non-null  int64  
dtypes: float64(3), int64(4), str(2)
memory usage: 8.0 MB
None
gender                 0
age                    0
hypertension           0
heart_disease          0
smoking_history        0
bmi                    0
HbA1c_level            0
blood_glucose_level    0
diabetes               0
dtype: int64
3854
diabetes
0    91500
1  

In [4]:
df = df.drop_duplicates()

In [5]:
X = df.drop('diabetes', axis=1)
y = df['diabetes']

In [6]:
X = pd.get_dummies(
    X,
    columns=['gender','smoking_history'],
    drop_first=False
)

feature_columns = X.columns.tolist()

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [8]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC(probability=True),
    'Naive Bayes': GaussianNB()
}

results=[]

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    pred=model.predict(X_test_scaled)

    results.append({
        'Model':name,
        'Accuracy':accuracy_score(y_test,pred),
        'Precision':precision_score(y_test,pred),
        'Recall':recall_score(y_test,pred),
        'F1 Score':f1_score(y_test,pred)
    })

results_df=pd.DataFrame(results).sort_values(by='Accuracy',ascending=False)
results_df

C:\Users\CodeWithPranav\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,Model,Accuracy,Precision,Recall,F1 Score
2,Random Forest,0.969527,0.949757,0.691038,0.800000
4,SVM,0.961726,0.967836,0.585495,0.729611
0,Logistic Regression,0.959594,0.868484,0.638561,0.735984
3,KNN,0.959178,0.886344,0.616156,0.726957
1,Decision Tree,0.947946,0.691039,0.741156,0.715220
5,Naive Bayes,0.292668,0.110711,0.998231,0.199317


In [10]:
best_model_name = results_df.iloc[0]['Model']
best_model = models[best_model_name]

print(best_model_name)

Random Forest


In [11]:
MODEL_DIR = Path('../trained_models')
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(best_model, MODEL_DIR/'diabetes_model.pkl')
joblib.dump(scaler, MODEL_DIR/'diabetes_scaler.pkl')
joblib.dump(feature_columns, MODEL_DIR/'diabetes_columns.pkl')

print('Saved successfully')

Saved successfully


In [12]:
model = joblib.load('../trained_models/diabetes_model.pkl')
scaler = joblib.load('../trained_models/diabetes_scaler.pkl')
columns = joblib.load('../trained_models/diabetes_columns.pkl')

sample = pd.DataFrame([{
    'gender':'Male',
    'age':25,
    'hypertension':0,
    'heart_disease':0,
    'smoking_history':'never',
    'bmi':22.5,
    'HbA1c_level':5.2,
    'blood_glucose_level':95
}])

sample = pd.get_dummies(sample, columns=['gender','smoking_history'])
sample = sample.reindex(columns=columns, fill_value=0)
sample = scaler.transform(sample)

print(model.predict(sample))
print(model.predict_proba(sample))

[0]
[[1. 0.]]
